## 1. Colab Environment Setup
Run in a personal Colab GPU runtime. Mount Drive when Colab asks you to authorize it.


In [ ]:
from pathlib import Path
import shutil
import torch

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = None
    print("Local mode: Colab Drive is unavailable.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name, "VRAM GiB:", round(gpu.total_memory / 2**30, 2))
else:
    print("No CUDA GPU. Formal training and Test evaluation will stop.")
print("RAM GiB:", round(__import__("psutil").virtual_memory().total / 2**30, 2))
print("Free disk GiB:", round(shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 2**30, 2))

## 2. Repository and Dependencies
Anthony's PR #2 is not merged. Pin the tested commit; do not write to GitHub.


In [ ]:
import subprocess
import sys

CODE_COMMIT = "f96ffa0f56ba9c5bee2ce51b6f28f84e630deab0"
REPO = Path("/content/PyHealth") if Path("/content").exists() else Path.cwd()

if Path("/content").exists():
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", "resnet-dnn-ecg",
                        "https://github.com/A-Wu5/PyHealth.git", str(REPO)], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_COMMIT], check=True)
    actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    if actual_commit != CODE_COMMIT:
        raise RuntimeError(f"/content/PyHealth is at {actual_commit}; expected {CODE_COMMIT}. Use a fresh runtime.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "wfdb"], check=True)
else:
    actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("PyHealth commit:", actual_commit)
assert actual_commit == CODE_COMMIT
sys.path.insert(0, str(REPO))

## 3. Data Preparation
Place an extracted Georgia folder containing E00001.mat and the chosen georgia_preprocessing*.zip in your private MyDrive. If several matches exist, set the paths below explicitly.


In [ ]:
import hashlib
import zipfile
import numpy as np
import wfdb

DATA_ROOT = Path("/content/data") if Path("/content").exists() else REPO / "data"
PROCESSED_ROOT = DATA_ROOT / "georgia_processed"
if DRIVE_ROOT is not None:
    RAW_GEORGIA_DIR = DRIVE_ROOT / "CS598/Georgia"
    PREPROCESSING_ZIP = DRIVE_ROOT / "CS598/georgia_preprocessing_v2.zip"
    # Update this path when the team provides corrected statistics.
else:
    RAW_GEORGIA_DIR = DATA_ROOT / "raw/G12EC/Georgia"
    PREPROCESSING_ZIP = DATA_ROOT.resolve().parent.parent / "georgia_preprocessing_v2.zip"

assert (RAW_GEORGIA_DIR / "E00001.mat").exists(), RAW_GEORGIA_DIR
assert PREPROCESSING_ZIP.exists(), PREPROCESSING_ZIP
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
expected_files = ["label_order.json", "split_seed1.npz", "normalization_seed1.npz"]
with zipfile.ZipFile(PREPROCESSING_ZIP) as archive:
    assert set(archive.namelist()) == set(expected_files)
    for name in expected_files:
        contents = archive.read(name)
        target = PROCESSED_ROOT / name
        if not target.exists() or target.read_bytes() != contents:
            target.write_bytes(contents)  # Only the private /content copy is changed.

with np.load(PROCESSED_ROOT / "split_seed1.npz") as split:
    record_ids = split["record_ids"].copy()
    header_paths = split["header_paths"].copy()
    train_idx = split["train_idx"].copy()
    assert len(record_ids) == len(header_paths) == 10344
    assert len(train_idx) == 8260

# Use the saved path for each record; never infer its group from the ID.
for record_id, saved_path in zip(record_ids, header_paths):
    record_id = str(record_id)
    parts = Path(str(saved_path)).parts
    assert parts[-3] == "georgia" and parts[-1] == record_id + ".hea"
    group = parts[-2]
    folder = DATA_ROOT / "georgia" / group
    folder.mkdir(parents=True, exist_ok=True)
    source_mat = RAW_GEORGIA_DIR / (record_id + ".mat")
    source_hea = RAW_GEORGIA_DIR / (record_id + ".hea")
    assert source_mat.exists() and source_hea.exists(), record_id
    target_mat = folder / source_mat.name
    if not target_mat.exists():
        target_mat.symlink_to(source_mat)
    header = source_hea.read_text()
    first, separator, remainder = header.partition("\n")
    fields = first.split(maxsplit=1)
    if fields[0] == record_id + ".mat":
        first = record_id + first[len(record_id) + 4:]
    elif fields[0] != record_id:
        raise ValueError(f"Unexpected header record name: {record_id}")
    corrected = first + (separator + remainder if separator else "")
    target_hea = folder / source_hea.name
    if not target_hea.exists() or target_hea.read_text() != corrected:
        target_hea.write_text(corrected)  # Never edit the source header.

first_record = Path(str(header_paths[0]))
example = wfdb.rdrecord(str(DATA_ROOT / "georgia" / first_record.parent.name / str(record_ids[0])))
assert example.fs == 500 and example.n_sig == 12 and all(unit == "mV" for unit in example.units)
print("Prepared", len(record_ids), "recordings from", RAW_GEORGIA_DIR)
print("Example:", example.p_signal.shape, example.sig_name)
print("Preprocessing ZIP:", PREPROCESSING_ZIP.name)

## 4. Imports and Configuration


In [ ]:
import json
import os
import random
import time
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from pyhealth.tasks.georgia_ecg import load_georgia_data
from pyhealth.models.resnet1d_18 import ResNet1D18

SEED = 1
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 8
MAX_EPOCHS = 250
VALIDATE_EVERY = 5
PATIENCE = 5
LEARNING_RATE = 0.001
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
train_generator = torch.Generator().manual_seed(SEED)
EXPERIMENT_DIR = DRIVE_ROOT / "CS598/experiment_seed1" if DRIVE_ROOT is not None else Path("/private/tmp/experiment_seed1_local")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Experiment directory:", EXPERIMENT_DIR)

## 5. Dataset and DataLoader


In [ ]:
train, val, test, codes, samples = load_georgia_data(
    data_root=DATA_ROOT, processed_root=PROCESSED_ROOT, seed=SEED
)
assert (len(train), len(val), len(test), len(codes)) == (8260, 1038, 1046, 30)
train_loader = DataLoader(train, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                          num_workers=0, generator=train_generator)
val_loader = DataLoader(val, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)
train_batch = next(iter(train_loader))
val_batch = next(iter(val_loader))
assert train_batch["signal"].shape[1:] == (12, 1250)
assert val_batch["signal"].shape[1:] == (8, 12, 1250)
assert train_batch["labels"].shape[1:] == val_batch["labels"].shape[1:] == (len(codes),)
print("Splits:", len(train), len(val), len(test))
print("Train batch:", train_batch["signal"].shape, train_batch["labels"].shape)
print("Validation batch:", val_batch["signal"].shape, val_batch["labels"].shape)

## 6. Normalization Verification
Compute full-length WFDB training moments using the loader's NaN handling and left padding. Formal training stops if the ZIP disagrees.


In [ ]:
with np.load(PROCESSED_ROOT / "normalization_seed1.npz") as saved:
    saved_mean = saved["mean"].copy()
    saved_std = saved["std"].copy()
    saved_train_idx = saved["train_idx"].copy()
    lead_names = saved["lead_names"].tolist()
assert np.array_equal(saved_train_idx, train_idx)
assert saved_mean.shape == saved_std.shape == (12,) and np.all(saved_std > 0)
total = np.zeros(12, dtype=np.float64)
squares = np.zeros(12, dtype=np.float64)
sample_count = 0
for i in train_idx:
    record_id = str(record_ids[i])
    group = Path(str(header_paths[i])).parent.name
    record = wfdb.rdrecord(str(DATA_ROOT / "georgia" / group / record_id))
    assert record.fs == 500 and record.n_sig == 12 and record.sig_name == lead_names
    signal = record.p_signal.T.copy()
    signal[np.isnan(signal)] = 0
    assert np.isfinite(signal).all() and signal.shape[1] <= 5000
    signal = np.pad(signal, ((0, 0), (5000 - signal.shape[1], 0)))
    total += signal.sum(axis=1)
    squares += np.square(signal).sum(axis=1)
    sample_count += signal.shape[1]
actual_mean = total / sample_count
actual_std = np.sqrt(squares / sample_count - actual_mean ** 2)
normalized_mean = (actual_mean - saved_mean) / saved_std
normalized_std = actual_std / saved_std
NORMALIZATION_OK = (np.allclose(actual_mean, saved_mean, rtol=1e-5, atol=1e-7)
                    and np.allclose(actual_std, saved_std, rtol=1e-5, atol=1e-7)
                    and np.max(np.abs(normalized_mean)) < 0.01
                    and np.max(np.abs(normalized_std - 1)) < 0.01)
print("Saved mean:", saved_mean)
print("WFDB training mean:", actual_mean)
print("Saved std:", saved_std)
print("WFDB training std:", actual_std)
print("Normalized training mean:", normalized_mean)
print("Normalized training std:", normalized_std)
# A cache from an older task must not supply different normalized samples.
cached = train.samples[0]
cached_id = cached["record_id"]
saved_position = np.where(record_ids == cached_id)[0][0]
cached_group = Path(str(header_paths[saved_position])).parent.name
fresh = wfdb.rdrecord(str(DATA_ROOT / "georgia" / cached_group / cached_id)).p_signal.T.copy()
fresh[np.isnan(fresh)] = 0
fresh = np.pad(fresh, ((0, 0), (5000 - fresh.shape[1], 0)))
fresh = ((fresh - saved_mean[:, None]) / saved_std[:, None]).astype(np.float32)
assert np.allclose(cached["signal"].numpy(), fresh, atol=1e-5)
print("Task cache matches current ZIP normalization: PASS")
print("Normalization:", "PASS" if NORMALIZATION_OK else "FAIL — formal training is blocked")

## 7. Model Initialization


In [ ]:
model = ResNet1D18(dataset=samples).to(device)
assert list(model.label_processor.label_vocab) == list(codes)
assert model.label_processor.output_size() == len(codes)
print("Model label order matches all", len(codes), "SNOMED codes.")

## 8. Prediction and Aggregation Functions


In [ ]:
def predict_dataset(model, loader):
    all_probabilities, all_labels, all_ids = [], [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch_size, windows = batch["signal"].shape[:2]
            signal = batch["signal"].reshape(batch_size * windows, 12, 1250).to(device)
            y_prob = model(signal=signal)["y_prob"]
            all_probabilities.append(y_prob.reshape(batch_size, windows, len(codes)).cpu())
            all_labels.append(batch["labels"].cpu())
            all_ids.extend(batch["record_id"])
    probabilities = torch.cat(all_probabilities)
    labels = torch.cat(all_labels).numpy()
    assert len(all_ids) == len(set(all_ids)) == len(loader.dataset)
    assert all_ids == [loader.dataset[i]["record_id"] for i in range(len(loader.dataset))]
    assert probabilities.shape == (len(all_ids), 8, len(codes))
    assert labels.shape == (len(all_ids), len(codes))
    assert np.isin(labels, [0, 1]).all()
    assert torch.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()
    return probabilities, labels, np.asarray(all_ids)

def aggregate_predictions(probabilities):
    return {
        "Max": probabilities.max(dim=1).values,
        "Mean": probabilities.mean(dim=1),
        "Top-2 Mean": probabilities.topk(2, dim=1).values.mean(dim=1),
    }

## 9. Evaluation Metrics


In [ ]:
def evaluate_auc(y_true, probabilities, codes):
    scores = np.asarray(probabilities)
    assert scores.shape == y_true.shape == (len(y_true), len(codes))
    assert np.isin(y_true, [0, 1]).all() and np.isfinite(scores).all()
    per_class = pd.Series(
        [roc_auc_score(y_true[:, i], scores[:, i]) for i in range(len(codes))],
        index=codes, name="ROC-AUC"
    )
    assert len(per_class) == len(codes) and np.isfinite(per_class).all()
    return per_class.mean(), per_class

toy = torch.tensor([[[0.1, 0.9], [0.8, 0.2], [0.6, 0.7]]])
toy_results = aggregate_predictions(toy)
assert torch.allclose(toy_results["Max"], torch.tensor([[0.8, 0.9]]))
assert torch.allclose(toy_results["Mean"], torch.tensor([[0.5, 0.6]]))
assert torch.allclose(toy_results["Top-2 Mean"], torch.tensor([[0.7, 0.8]]))
assert all(value.shape == (1, 2) for value in toy_results.values())
toy_labels = np.array([[0, 1], [1, 0], [0, 1], [1, 0]])
toy_scores = np.array([[0.1, 0.9], [0.9, 0.1], [0.2, 0.8], [0.8, 0.2]])
assert evaluate_auc(toy_labels, toy_scores, ["a", "b"])[0] == 1.0
print("Aggregation and ROC-AUC numerical checks: PASS")

## 10. Training Smoke Test
The small-batch test runs on available hardware. Formal training remains gated on CUDA and verified normalization.


In [ ]:
small_signal = train_batch["signal"][:2].to(device)
small_labels = train_batch["labels"][:2].to(device)
smoke_optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model.train()
smoke_optimizer.zero_grad()
smoke_output = model(signal=small_signal, labels=small_labels)
assert smoke_output["logit"].shape == smoke_output["y_prob"].shape == (2, len(codes))
assert torch.isfinite(smoke_output["loss"])
smoke_output["loss"].backward()
gradient = model.backbone.stem[0].weight.grad
assert gradient is not None and torch.isfinite(gradient).all() and gradient.abs().sum() > 0
before = model.fc2.weight.detach().clone()
smoke_optimizer.step()
assert not torch.equal(before, model.fc2.weight)
print("Forward/loss/backward/optimizer step: PASS; loss:", float(smoke_output["loss"].detach()))
print("Device:", device)
if device.type == "cuda":
    print("Peak VRAM GiB:", round(torch.cuda.max_memory_allocated() / 2**30, 2))

# A temporary checkpoint checks the save/load interface.
import tempfile
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "smoke.ckpt"
    torch.save(model.state_dict(), path)
    restored = ResNet1D18(dataset=samples).to(device)
    restored.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    assert torch.equal(restored.fc2.weight, model.fc2.weight)
print("Checkpoint save/load: PASS")
model.eval()
with torch.no_grad():
    batch_prob = model(signal=val_batch["signal"][:1].reshape(8, 12, 1250).to(device))["y_prob"]
assert aggregate_predictions(batch_prob.reshape(1, 8, len(codes)))["Max"].shape == (1, len(codes))
print("Window inference and Max aggregation: PASS")

# Reset all training state after the smoke test.
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
train.rng = np.random.default_rng(SEED)
train_generator.manual_seed(SEED)
del smoke_optimizer, smoke_output, restored
model = ResNet1D18(dataset=samples).to(device)
if device.type == "cuda":
    torch.cuda.empty_cache()

## 11. Training and Validation
The first formal epoch reports its training time. The default follows the paper's 250-epoch cap. Resume uses last.ckpt when present.


In [ ]:
assert actual_commit == CODE_COMMIT
normalization_hash = hashlib.sha256((PROCESSED_ROOT / "normalization_seed1.npz").read_bytes()).hexdigest()
split_hash = hashlib.sha256((PROCESSED_ROOT / "split_seed1.npz").read_bytes()).hexdigest()
config = {
    "code_commit": CODE_COMMIT, "seed": SEED, "codes": list(codes),
    "split_sha256": split_hash, "normalization_sha256": normalization_hash,
    "preprocessing_zip": PREPROCESSING_ZIP.name,
    "model": "ResNet1D18", "backbone_out_dim": 256, "hidden_dim": 128,
    "dropout": 0.25, "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "max_epochs": MAX_EPOCHS, "validate_every": VALIDATE_EVERY,
    "patience": PATIENCE, "selection_metric": "Validation Max Macro ROC-AUC",
}
config_path = EXPERIMENT_DIR / "config.json"
if config_path.exists() and (EXPERIMENT_DIR / "last.ckpt").exists():
    old_config = json.loads(config_path.read_text())
    for key in config:
        if key != "max_epochs":
            assert old_config[key] == config[key], f"Resume config mismatch: {key}"
best_path = EXPERIMENT_DIR / "best.ckpt"
last_path = EXPERIMENT_DIR / "last.ckpt"
history_path = EXPERIMENT_DIR / "training_history.csv"

def save_last(epoch, best_auc, bad_checks, history, optimizer):
    state = {
        "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "epoch": epoch, "best_auc": best_auc, "bad_checks": bad_checks,
        "history": history, "config": config,
        "train_window_rng": train.rng.bit_generator.state,
        "train_shuffle_rng": train_generator.get_state(),
        "torch_rng": torch.get_rng_state(), "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
    }
    temporary = last_path.with_suffix(".ckpt.tmp")
    torch.save(state, temporary)
    os.replace(temporary, last_path)

TRAINING_COMPLETED = False
if not NORMALIZATION_OK:
    print("Training skipped: saved normalization disagrees with WFDB training statistics.")
elif device.type != "cuda":
    print("Training skipped: CUDA GPU is required.")
else:
    config_path.write_text(json.dumps(config, indent=2))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_auc, bad_checks, start_epoch, history = -1.0, 0, 1, []
    if last_path.exists():
        state = torch.load(last_path, map_location=device, weights_only=False)
        for key in config:
            if key != "max_epochs":
                assert state["config"][key] == config[key], f"Checkpoint mismatch: {key}"
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        best_auc, bad_checks = state["best_auc"], state["bad_checks"]
        start_epoch, history = state["epoch"] + 1, state["history"]
        train.rng.bit_generator.state = state["train_window_rng"]
        train_generator.set_state(state["train_shuffle_rng"].cpu())
        torch.set_rng_state(state["torch_rng"].cpu())
        np.random.set_state(state["numpy_rng"])
        random.setstate(state["python_rng"])
        torch.cuda.set_rng_state_all([rng.cpu() for rng in state["cuda_rng"]])
        print("Resuming after epoch", state["epoch"])
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.train()
        epoch_start = time.time()
        total_loss = 0.0
        total_examples = 0
        for batch in train_loader:
            optimizer.zero_grad()
            output = model(signal=batch["signal"].to(device),
                           labels=batch["labels"].to(device))
            output["loss"].backward()
            optimizer.step()
            batch_size = batch["labels"].shape[0]
            total_loss += float(output["loss"].detach()) * batch_size
            total_examples += batch_size
        training_seconds = time.time() - epoch_start
        row = {"epoch": epoch, "training_loss": total_loss / total_examples,
               "validation_macro_auc": np.nan}
        if epoch == 1:
            print("First formal epoch seconds:", round(training_seconds, 1))
            print("Peak VRAM GiB:", round(torch.cuda.max_memory_allocated() / 2**30, 2))
        if epoch % VALIDATE_EVERY == 0 or epoch == MAX_EPOCHS:
            val_prob, val_labels, val_ids = predict_dataset(model, val_loader)
            val_auc, _ = evaluate_auc(val_labels, aggregate_predictions(val_prob)["Max"].numpy(), codes)
            row["validation_macro_auc"] = float(val_auc)
            if val_auc > best_auc:
                best_auc, bad_checks = float(val_auc), 0
                best_temporary = best_path.with_suffix(".ckpt.tmp")
                torch.save(model.state_dict(), best_temporary)
                os.replace(best_temporary, best_path)
                print("Best checkpoint: epoch", epoch, "AUC", round(best_auc, 5))
            else:
                bad_checks += 1
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        save_last(epoch, best_auc, bad_checks, history, optimizer)
        print("Epoch", epoch, "loss", round(row["training_loss"], 5),
              "val Max AUC", row["validation_macro_auc"], "bad checks", bad_checks)
        if bad_checks >= PATIENCE:
            print("Early stopping at epoch", epoch)
            break
    TRAINING_COMPLETED = best_path.exists()

## 12. Training History


In [ ]:
if history_path.exists():
    history_frame = pd.read_csv(history_path)
    print(history_frame.tail(10).to_string(index=False))
else:
    print("No formal training history yet.")

## 13. Final Test Evaluation
Run Test once with the best Validation checkpoint. This cell does not tune the model.


In [ ]:
if not NORMALIZATION_OK or device.type != "cuda" or not TRAINING_COMPLETED:
    print("Formal Test evaluation skipped until training completes with verified normalization and CUDA.")
else:
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    model.eval()
    test_probabilities, test_labels, test_ids = predict_dataset(model, test_loader)
    assert test_probabilities.shape == (1046, 8, len(codes))
    assert test_labels.shape == (1046, len(codes)) and len(test_ids) == 1046
    assert list(test_ids) == [test[i]["record_id"] for i in range(len(test))]
    print("Test predictions:", test_probabilities.shape, "labels:", test_labels.shape)

## 14. Aggregation Ablation and Results
All three methods use the same Test window probabilities. The paper reports G12EC ResNet-18 Macro ROC-AUC 0.979 ± 0.002 over five independent experiments; this notebook uses one fixed split.


In [ ]:
if not NORMALIZATION_OK or device.type != "cuda" or not TRAINING_COMPLETED:
    print("Ablation skipped: no eligible trained checkpoint.")
else:
    predictions = aggregate_predictions(test_probabilities)
    for values in predictions.values():
        assert values.shape == (len(test), len(codes)) and torch.isfinite(values).all()
    metrics = {name: evaluate_auc(test_labels, values.numpy(), codes)
               for name, values in predictions.items()}
    results = pd.DataFrame({
        "Aggregation": list(metrics),
        "Macro ROC-AUC": [metrics[name][0] for name in metrics],
    })
    results["Delta vs. Max"] = results["Macro ROC-AUC"] - results.loc[0, "Macro ROC-AUC"]
    per_class = pd.DataFrame({name: metrics[name][1] for name in metrics})
    per_class.index.name = "SNOMED code"
    results.to_csv(EXPERIMENT_DIR / "ablation_results.csv", index=False)
    per_class.to_csv(EXPERIMENT_DIR / "per_class_auc.csv")
    np.savez_compressed(EXPERIMENT_DIR / "test_predictions.npz",
                        window_probabilities=test_probabilities.numpy(),
                        labels=test_labels, record_ids=test_ids, codes=np.asarray(codes))
    import matplotlib.pyplot as plt
    history_frame = pd.read_csv(history_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history_frame["epoch"], history_frame["training_loss"])
    axes[0].set_title("Training loss")
    observed = history_frame.dropna(subset=["validation_macro_auc"])
    axes[1].plot(observed["epoch"], observed["validation_macro_auc"], marker="o")
    axes[1].set_title("Validation Max Macro ROC-AUC")
    axes[2].bar(results["Aggregation"], results["Macro ROC-AUC"])
    axes[2].set_title("Test aggregation comparison")
    fig.tight_layout()
    fig.savefig(EXPERIMENT_DIR / "experiment_plots.png", dpi=150)
    plt.show()
    print("Single-split Test results with the best Validation checkpoint:")
    print(results.to_string(index=False))
    print("Saved to:", EXPERIMENT_DIR)